In [1]:
import sys
import os

# Set working directory explicitly to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)

from src.nlp.news_fetcher import NewsFetcher
from src.nlp.sentiment_analyzer import SentimentAnalyzer
from src.data.spark_pipeline import get_spark_session
from src.data.databricks_client import DatabricksClient

print("Phase 2 imports successful!")

Phase 2 imports successful!


In [2]:
# 1. Fetch news articles
fetcher = NewsFetcher()
news_df = fetcher.fetch_ticker_news("AAPL", limit=10)

# 2. Analyze sentiment
analyzer = SentimentAnalyzer()
sentiment_df = analyzer.add_sentiment_features(news_df)

sentiment_df[["timestamp", "ticker", "title", "compound_score", "pos_score", "neg_score"]].head()

[2026-09-23 17:51:08] [INFO] [stonks_maker]: Fetching news for AAPL via yfinance fallback...
[2026-09-23 17:51:09] [INFO] [stonks_maker]: Computing sentiment scores for news dataset...
[2026-09-23 17:51:09] [INFO] [stonks_maker]: Sentiment scoring completed.


,timestamp,ticker,title,compound_score,pos_score,neg_score
0,2026-09-23 17:32:27+00:00,AAPL,Looking for consumer and investor sentiment tr...,0.5574,0.101,0.037
1,2026-09-23 13:53:40+00:00,AAPL,Why Apple's stock chart is probably putting a ...,0.3612,0.079,0.000
2,2026-09-23 10:00:00+00:00,AAPL,"The AI panic, tech concentration, and Nvidia w...",-0.5267,0.070,0.123
3,2026-09-22 19:59:00+00:00,AAPL,"AI can transform shopping, but human touch win...",0.6956,0.076,0.020
4,2026-09-22 17:54:01+00:00,AAPL,Apple doesn't need to be in first place for AI...,0.4939,0.048,0.000


In [3]:
# Convert sentiment dataset to Spark DataFrame
spark = get_spark_session()
spark_news_df = spark.createDataFrame(sentiment_df)

# Save to Parquet/Delta storage
db_client = DatabricksClient()
db_client.write_dataset(spark_news_df, table_name="aapl_sentiment")

print("Phase 2 Execution Complete!")

[2026-09-23 17:51:14] [INFO] [stonks_maker]: Databricks credentials not configured. Saving locally to Parquet.
[2026-09-23 17:51:16] [INFO] [stonks_maker]: Successfully saved to /home/jovyan/work/data/processed/aapl_sentiment
Phase 2 Execution Complete!
